In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from typing import Literal
import warnings

# Suppress warnings for cleaner output (optional)
warnings.filterwarnings('ignore', category=RuntimeWarning)

END_MIN = 10
SAMPLE_DATA_FILE = r"D:\Data\TCS\TCdata.xlsx"
METRIC_PREFIXES = ['Di', 'Do', 'Si', 'So']

SE_or_VG: Literal["SE", "VG", "Both"] = "Both"
DISABLE_ALL_FILTERS = False
PRE_BIAS_THRESH = 0.5

# ==========================================
# 1. LOAD & FILTER DATA
# ==========================================
df = pd.read_excel(SAMPLE_DATA_FILE, sheet_name='wt-total')

# Filter data if needed
if not DISABLE_ALL_FILTERS:
    if SE_or_VG == "SE":
        df = df[df["Virgin"] != 1]
    elif SE_or_VG == "VG":
        df = df[df["Virgin"] == 1]

    df['PB'] = (df['Di_pre'] + df['Do_pre'] - df['Si_pre'] - df['So_pre'])/(df['Di_pre'] + df['Do_pre'] + df['Si_pre'] + df['So_pre'])
    df = df[abs(df['PB']) <= PRE_BIAS_THRESH]

# ==========================================
# 2. CALCULATE PI FOR 0-10min
# ==========================================
def calculate_pi_10min(row):
    """Calculate PI = (Di_1-10min - Si_1-10min) / (Di_1-10min + Si_1-10min)"""
    di_cols = [f'Di_{i}min' for i in range(1, END_MIN+1)]
    si_cols = [f'Si_{i}min' for i in range(1, END_MIN+1)]
    
    di_sum = sum(row[col] for col in di_cols if col in row.index and not pd.isna(row[col]))
    si_sum = sum(row[col] for col in si_cols if col in row.index and not pd.isna(row[col]))
    
    if (di_sum + si_sum) == 0:
        return np.nan
    
    return (di_sum - si_sum) / (di_sum + si_sum)

df['PI_10min'] = df.apply(calculate_pi_10min, axis=1)

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def sem(x):
    """Calculate standard error of mean, handling NaNs."""
    x = np.asarray(x)
    x_clean = x[~np.isnan(x)]
    return np.std(x_clean, ddof=1) / np.sqrt(len(x_clean)) if len(x_clean) > 1 else np.nan

def format_pval(p):
    if pd.isna(p):
        return "p=NA"
    stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    if p >= 0.001:
        return f"{stars}\n(p={p:.3f})"
    else:
        return f"{stars}\n(p={p:.4f})"

def add_pval_annotation(ax, x1, x2, y_base, p_value, y_offset=0.1, height=0.05):
    """Add p-value annotation with bracket."""
    if pd.isna(p_value):
        return
    
    y_top = y_base * (1 + y_offset)
    label = format_pval(p_value)
    
    # Draw bracket
    ax.plot([x1, x1, x2, x2], [y_base, y_top, y_top, y_base], 
            lw=1.5, c='black', zorder=5)
    
    # Add text
    ax.text((x1 + x2) / 2, y_top + height, label, 
            ha='center', va='bottom', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.9),
            zorder=6)

# ==========================================
# 4. STATISTICAL TESTING (PI)
# ==========================================
print("\n=== One-sample t-tests: PI vs 0 by Cycle Stage ===")
Cycles = ['P', 'E', 'M', 'D']
pi_pvals_vs_zero = {}
pi_stats = {}

for stage in Cycles:
    mask = df['Cycle'] == stage
    pi_vals = df.loc[mask, 'PI_10min'].dropna()
    
    if len(pi_vals) > 1:
        t_stat, p_val = stats.ttest_1samp(pi_vals, popmean=0)
        pi_pvals_vs_zero[stage] = p_val
        pi_stats[stage] = {
            'mean': pi_vals.mean(),
            'sem': sem(pi_vals),
            'n': len(pi_vals),
            'values': pi_vals.values
        }
        direction = "D-pref" if pi_vals.mean() > 0 else "S-pref" if pi_vals.mean() < 0 else "neutral"
        print(f"{stage:5s} | Mean PI={pi_vals.mean():.3f} ({direction}) | t={t_stat:6.3f} | p={p_val:.4f} {format_pval(p_val).split()[0]} (n={len(pi_vals)})")
    else:
        pi_pvals_vs_zero[stage] = np.nan
        pi_stats[stage] = None

# Test E vs D groups
print("\n=== Welch's t-test: E vs D groups ===")
if pi_stats['E'] and pi_stats['D']:
    e_vals = pi_stats['E']['values']
    d_vals = pi_stats['D']['values']
    
    if len(e_vals) > 1 and len(d_vals) > 1:
        t_stat, p_val_ed = stats.ttest_ind(e_vals, d_vals, equal_var=False)
        print(f"E vs D | t={t_stat:6.3f} | p={p_val_ed:.4f} {format_pval(p_val_ed).split()[0]}")
    else:
        p_val_ed = np.nan
else:
    p_val_ed = np.nan

# ==========================================
# 5. CREATE FIRST BAR PLOT (PI)
# ==========================================
fig, ax = plt.subplots(figsize=(10, 8))

stage_colors = {
    'P': '#FFF351',  # 
    'E': '#FA1313',  # 
    'M': '#65EE75',  #  
    'D': '#0C9DDB'   #  
}

bar_width = 0.6
x_positions = range(len(Cycles))

for i, stage in enumerate(Cycles):
    if pi_stats[stage] is None:
        continue
    
    mean_pi = pi_stats[stage]['mean']
    sem_pi = pi_stats[stage]['sem']
    values = pi_stats[stage]['values']
    
    ax.bar(i, mean_pi, width=bar_width, color=stage_colors[stage], edgecolor='black', linewidth=1.5, alpha=0.7, zorder=2)
    ax.errorbar(i, mean_pi, yerr=sem_pi, fmt='none', ecolor='black', capsize=8, capthick=2, linewidth=2, zorder=3)
    
    jitter = np.random.normal(0, 0.08, size=len(values))
    ax.scatter(i + jitter, values, c=stage_colors[stage], s=60, alpha=0.8, edgecolors='black', linewidth=0.8, zorder=4)
    
    p_val = pi_pvals_vs_zero.get(stage)
    if p_val is not None and not np.isnan(p_val):
        y_base = max(0, mean_pi) + sem_pi + 0.05
        add_pval_annotation(ax, i - 0.15, i + 0.15, y_base, p_val, y_offset=0.15, height=0.03)

if not np.isnan(p_val_ed):
    e_idx = Cycles.index('E')
    d_idx = Cycles.index('D')
    e_mean = pi_stats['E']['mean']
    d_mean = pi_stats['D']['mean']
    y_base = max(e_mean, d_mean) + max(pi_stats['E']['sem'], pi_stats['D']['sem']) + 0.1
    add_pval_annotation(ax, e_idx - 0.2, d_idx + 0.2, y_base, p_val_ed, y_offset=0.2, height=0.03)

ax.set_xticks(x_positions)
ax.set_xticklabels(Cycles, fontsize=14, fontweight='bold')
ax.set_ylabel(f'Preference Index (0-{END_MIN} min)\n(Di-Si)/(Di+Si)', fontsize=12, fontweight='bold')
ax.set_title('Social Preference by Estrous Cycle Stage', fontsize=16, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.8)
ax.axhline(0, color='black', linewidth=1.5, linestyle='-', zorder=1)
ax.set_ylim(-0.6, 0.6)

for i, stage in enumerate(Cycles):
    if pi_stats[stage]:
        n = pi_stats[stage]['n']
        ax.text(i, -0.52, f'n={n}', ha='center', va='top', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


# ==============================================================================
# NEW SECTIONS BELOW: GENERAL SOCIOSEXUAL INTEREST
# ==============================================================================

# ==========================================
# 6. CALCULATE GENERAL SOCIOSEXUAL INTEREST
# ==========================================
def calculate_general_interest(row):
    """Calculate general sociosexual interest = (Sum of Di_1-END_MIN + Sum of Si_1-END_MIN) / END_MIN"""
    di_cols = [f'Di_{i}min' for i in range(1, END_MIN+1)]
    si_cols = [f'Si_{i}min' for i in range(1, END_MIN+1)]
    
    di_sum = sum(row[col] for col in di_cols if col in row.index and not pd.isna(row[col]))
    si_sum = sum(row[col] for col in si_cols if col in row.index and not pd.isna(row[col]))
    
    return (di_sum + si_sum) / ( END_MIN * 0.6 )

df['General_Interest'] = df.apply(calculate_general_interest, axis=1)

# ==========================================
# 7. STATISTICAL TESTING FOR GENERAL INTEREST
# ==========================================
print("\n=== General Social Interest by Cycle Stage ===")
interest_stats = {}
interest_vals_by_stage = {}

for stage in Cycles:
    mask = df['Cycle'] == stage
    vals = df.loc[mask, 'General_Interest'].dropna()
    interest_stats[stage] = {
        'mean': vals.mean(),
        'sem': sem(vals),
        'n': len(vals),
        'values': vals.values
    }
    interest_vals_by_stage[stage] = vals
    print(f"{stage:5s} | Mean Interest={vals.mean():.3f} | SEM={sem(vals):.3f} (n={len(vals)})")

# Test E vs D groups for General Interest (Replacing ANOVA)
print("\n=== Welch's t-test: E vs D groups (General Interest) ===")
e_vals_int = interest_vals_by_stage['E']
d_vals_int = interest_vals_by_stage['D']

if len(e_vals_int) > 1 and len(d_vals_int) > 1:
    t_stat_int, p_val_ed_int = stats.ttest_ind(e_vals_int, d_vals_int, equal_var=False)
    print(f"E vs D | t={t_stat_int:6.3f} | p={p_val_ed_int:.4f} {format_pval(p_val_ed_int).split()[0]}")
else:
    p_val_ed_int = np.nan

# ==========================================
# 8. CREATE SECOND BAR PLOT: GENERAL INTEREST
# ==========================================
fig2, ax2 = plt.subplots(figsize=(10, 8))

bar_width = 0.6
x_positions = range(len(Cycles))

# Calculate max y for dynamic scaling
max_y = 0.1  # baseline to prevent zero-division or flat axes
for stage in Cycles:
    if interest_stats[stage] and not np.isnan(interest_stats[stage]['mean']):
        current_max = interest_stats[stage]['mean'] + 3 * interest_stats[stage]['sem']
        if current_max > max_y:
            max_y = current_max

# Plot bars and individual points
for i, stage in enumerate(Cycles):
    if interest_stats[stage] is None or len(interest_stats[stage]['values']) == 0:
        continue
    
    mean_int = interest_stats[stage]['mean']
    sem_int = interest_stats[stage]['sem']
    values = interest_stats[stage]['values']
    
    # Bar
    ax2.bar(i, mean_int, width=bar_width, 
           color=stage_colors[stage], 
           edgecolor='black', linewidth=1.5,
           alpha=0.7, zorder=2)
    
    # Error bar
    ax2.errorbar(i, mean_int, yerr=sem_int, fmt='none', 
                ecolor='black', capsize=8, capthick=2, 
                linewidth=2, zorder=3)
    
    # Individual data points with jitter
    jitter = np.random.normal(0, 0.08, size=len(values))
    ax2.scatter(i + jitter, values, 
               c=stage_colors[stage], s=60, 
               alpha=0.8, edgecolors='black', linewidth=0.8, 
               zorder=4)
    
    # Sample size annotation
    n = interest_stats[stage]['n']
    ax2.text(i, 25, f'n={n}', ha='center', va='top', 
            fontsize=10, fontweight='bold')

# Add E vs D p-value annotation (Replacing the ANOVA spanning bracket)
if not np.isnan(p_val_ed_int):
    e_idx = Cycles.index('E')
    d_idx = Cycles.index('D')
    e_mean_int = interest_stats['E']['mean']
    d_mean_int = interest_stats['D']['mean']
    
    # Calculate y_base dynamically based on the highest of the two bars + SEM
    y_base = max(e_mean_int, d_mean_int) + max(interest_stats['E']['sem'], interest_stats['D']['sem']) + 0.05 * max_y
    
    # Use the helper function to draw the bracket between E and D
    add_pval_annotation(ax2, e_idx - 0.2, d_idx + 0.2, y_base, p_val_ed_int, 
                        y_offset=0.15, height=0.04 * max_y)
    
    # Adjust y-limit if needed so the bracket isn't cut off
    needed_y = y_base * (1 + 0.15) + 0.06 * max_y
    if needed_y > max_y * 1.25:
        ax2.set_ylim(30, needed_y * 1.1)
    else:
        ax2.set_ylim(30, max_y * 1.25)
else:
    ax2.set_ylim(30, max_y * 1.25)

# Formatting
ax2.set_xticks(x_positions)
ax2.set_xticklabels(Cycles, fontsize=14, fontweight='bold')
ax2.set_ylabel(f'General Sociosexual Interest\n(%)', fontsize=12, fontweight='bold')
ax2.set_title('General Sociosexual Interest by Estrous Cycle Stage', fontsize=16, fontweight='bold', pad=20)
ax2.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.show()

print("\n✅ Second figure created with General Sociosexual Interest grouped by Cycle stages.")
print("   - Welch's t-test comparing Estrus (E) vs Diestrus (D)")